主要实现功能：整理库存数据

In [1]:
import pandas as pd
from functools import reduce

import openpyxl
import warnings


pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

In [2]:
date_parm = '20251202'

In [3]:
fba_path = f'../src_data/FBA库存/FBA库存-自定义导出-{date_parm}.xlsx'

product_path = f'../src_data/产品库存/产品库存-自定义导出-{date_parm}.xlsx'

yuzhan_path = f'../src_data/预占单/调拨单20250325.xlsx'  # 积加上下载的未出库状态的调拨单

In [4]:
# usecols_fba = ['仓库', 'SKU', 'MSKU', 
#                 '已发货(数量)', '正在接收(数量)',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
#                 '预留-运营中心转运(数量)', '预留-运营中心正在处理(数量)', 'FBA可售(数量)', '可用库存(数量)',  # 可用库存(数量) = FBA可售(数量) + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量)
#                 '271~330(数量)', '331~365(数量)', '365+(数量)', 
#                 '7天日均', '15天日均', '30天日均']
usecols_fba = ['仓库', 'SKU', 'MSKU', 
                '货件已发货', '货件正在接收',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
                '预留-运营中心转运', '预留-运营中心正在处理', '本地可售', '可用量',  # 可用量 = 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售
                '271~330', '331~365', '365+', 
                '平台7天日均', '平台15天日均', '平台30天日均']

usecols_pro = ['仓库', '平台站点', 'SKU', 'MSKU',
                '采购量',  # 采购量  = 采购单未交货数量 = 已下单数量
                '在途量',  # 在途量 = 已发货量 + 待入库量 / 在途量 = 本地-在途 
                '可用量']


fba_df = pd.read_excel(io=fba_path, usecols=usecols_fba)
fba_df.rename(columns={'平台7天日均': '7天日均', '平台15天日均': '15天日均', '平台30天日均': '30天日均'}, inplace=True)
fba_df.rename(columns={'货件已发货': '已发货(数量)', '本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)

product_df = pd.read_excel(io=product_path, usecols=usecols_pro)
# product_df.rename(columns={'SKU0001': 'SKU', '采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)  # todo
product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

# 这段代码会找到所有“仓库”为 "TK本地仓" 且“站点”为 "共享" 的行，并将其“站点”值改为 "TIKTOK:TK本地仓:TK"
product_df.loc[(product_df['仓库'] == 'TK本地仓') & (product_df['站点'] == '共享'), '站点'] = 'TIKTOK:TK本地仓:TK本地仓'

yuzhan_df = pd.read_excel(yuzhan_path, sheet_name='明细数据', usecols=['SKU', 'MSKU', '调出数量', '调入仓', '单据状态', '拣货状态', '出库状态'])

In [5]:
# product_df = pd.read_excel(io=product_path, usecols=usecols_pro)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

In [6]:
# product_df = new_product_df

In [7]:
product_df.columns

Index(['仓库', 'MSKU', '站点', 'SKU', '已下单数量', '本地-在途', '本地'], dtype='object')

# 1、预处理FBA库存数据

In [8]:
# 去掉二次销售MSKU和MX站点的数据
fba_process = fba_df[~(fba_df.MSKU.str.contains('amzn') | fba_df.仓库.str.contains('MX_FBA'))]

# 格式化仓库 shop:site
fba_process['仓库'] = fba_process.仓库.str.split('_FBA', expand=True)[0]

# 把EU站点映射成DE
fba_process['仓库'] = fba_process['仓库'].apply(lambda x: x[:-2] + 'DE' if 'EU' in x else x)

# 获取站点
fba_process['站点'] = fba_process.仓库.str[-2:]

# 定义 长期仓储库存
fba_process['长期仓储库存'] = fba_process[['271~330', '331~365', '365+']].sum(axis=1)
fba_process.drop(columns=['271~330', '331~365', '365+'], inplace=True)

fba_process.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存
2065,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,0,0,35,10,1171,1216,17.00,16.27,14.57,US,0
3190,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,CA,0
4128,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,JP,0
4129,SEEKWAY:JP,SP001-406 Black 40 NEW,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,JP,0
6809,SENWAYZON:DE,SP001EU-406-Black-41,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,DE,0
14768,SENWAYZON:UK,SP001EU-406-Black-40-new,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,UK,0
15551,SEEKWAY:CA,SP001-406 Black New 40,SP001-406 Black 40,59,0,21,42,72,135,2.00,1.33,1.10,CA,0
19542,SEEKWAY:JP,JPSP001-406 Black 40,SP001-406 Black 40,0,0,0,0,40,40,0.43,0.20,0.20,JP,0


In [9]:
fba_process.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存'],
      dtype='object')

In [10]:
fba_process.仓库.unique()

array(['rivbos:US', 'rivbos:CA', 'rivbos:UK', 'rivbos:JP', 'rivbos:DE',
       'SEEKWAY:US', 'SEEKWAY:CA', 'SEEKWAY:JP', 'SIMARI:JP',
       'SENWAYZON:US', 'SENWAYZON:CA', 'SENWAYZON:UK', 'SENWAYZON:DE',
       'SAYOLA:US', 'SAYOLA:CA', 'RIVMOUNT:US', 'RIVMOUNT:CA',
       'SIMARI:US', 'SIMARI:CA', 'SIMARI:UK', 'SIMARI:DE', 'ROUNDYAP:US',
       'Mr_y:US', 'GLOW:US', 'CIKERWEL:US', 'WAYFINDING:US',
       'WAYFINDING:CA', 'GLOW:CA', 'CIKERWEL:CA'], dtype=object)

In [11]:
fba_process.站点.unique()

array(['US', 'CA', 'UK', 'JP', 'DE'], dtype=object)

# 2、预处理海外仓库存和本地仓库存

In [12]:
# # 新增 站点 列，根据 仓库 列是否包含 "FBA" 来赋值
# product_df['站点'] = product_df['仓库'].apply(lambda x: x if 'FBA' in str(x) else '共享').str.replace('_FBA', '', regex=False)
# # 新增 店铺 列
# product_df['店铺'] = product_df['站点'].str.replace('EU', 'DE', regex=False)
# # 构建 站点
# product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x).str.replace('EU', 'DE', regex=False)

In [13]:
product_df.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '元坤海外仓', '棉帽-广州仓', '小仓库旧货-本地仓',
       '其它品类-广州仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'CN', 'B端本地仓', 'US易速达海外仓', 'UK商易海外仓',
       'TK本地仓', '易速达2(废弃）:US-GA-1', '异常仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       'IT永翔海外仓'], dtype=object)

In [14]:
'''
# todo：枚举法排除
# 去除TIKTOK数据
product_df = product_df[~product_df['站点'].str.contains('TIKTOK')]
# 去掉TK仓
product_df = product_df[~product_df['仓库'].str.contains('TK')]

# 去除三方仓数据
product_df = product_df[~product_df['仓库'].str.contains('GA')]

# 去除异常仓 todo
product_df = product_df[~product_df['仓库'].str.contains('异常')]
'''

product_df = product_df[product_df['仓库'].str.contains('广州仓|海外仓|易速达美东GA仓|B端|TK本地仓')]

# 格式化站点 shop:site
product_df['店铺'] = product_df.站点.apply(lambda x: x if x == '共享' else x[len('AMAZON:'):])

# 构建 站点
product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x)


In [15]:
product_df.店铺.unique()

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW:US',
       'RIVMOUNT:US', 'ROUNDYAP:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'WAYFINDING:US', 'SENWAYZON:US', 'SAYOLA:US',
       'SENWAYZON:CA', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING:CA',
       'SIMARI:CA', 'GLOW:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [16]:
product_df.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '元坤海外仓', '棉帽-广州仓', '其它品类-广州仓',
       '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓', 'DE商易海外仓',
       'US商易海外仓', 'B端本地仓', 'US易速达海外仓', 'UK商易海外仓', 'TK本地仓',
       'CN易速达:易速达美东GA仓', 'IT永翔海外仓'], dtype=object)

In [17]:
product_df.站点.unique()

array(['共享', 'DE', 'US', 'UK', 'JP', 'CA', 'United_States', 'TK本地仓'],
      dtype=object)

In [18]:
product_df.店铺.unique()

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW:US',
       'RIVMOUNT:US', 'ROUNDYAP:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'WAYFINDING:US', 'SENWAYZON:US', 'SAYOLA:US',
       'SENWAYZON:CA', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING:CA',
       'SIMARI:CA', 'GLOW:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [19]:
# # 使用正则表达式一次性排除 '海外' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
# pattern = r'^(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓)(?!.*TK本地仓).*'
# local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
# local_inv.仓库.unique()

In [20]:
# 使用正则表达式一次性排除 '海外' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
pattern = r'^(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓).*'
local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
local_inv.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '棉帽-广州仓', '其它品类-广州仓', 'TK本地仓'],
      dtype=object)

In [21]:
local_inv

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,店铺
0,水鞋-广州仓,WP002-208 Black 47,共享,WP002-208 Black 47,0,0,0,共享
1,水鞋-广州仓,SWS005-501 circular black 40-41,共享,SWS005-501 circular black 40-41,0,0,0,共享
2,眼镜-广州仓,RBK004-1 Black Coating Lens,共享,RBK004-1 Black Coating Lens,0,0,0,共享
3,水鞋-广州仓,SP001-407 White 39,共享,SP001-407 White 39,0,0,0,共享
4,水鞋-广州仓,SP001-407 White 41,共享,SP001-407 White 41,0,0,0,共享
...,...,...,...,...,...,...,...,...
105845,水鞋-广州仓,SP001-CA 412 Light Grey 42,CA,SP001-412 Light Grey 42,0,0,0,SEEKWAY:CA
105846,水鞋-广州仓,SP001-CA 412 Light Grey 44,CA,SP001-412 Light Grey 44,0,0,0,SEEKWAY:CA
105847,水鞋-广州仓,SP001-CA 412 Light Grey 43,CA,SP001-412 Light Grey 43,0,0,0,SEEKWAY:CA
105848,手套-广州仓,RG803 Black-Gray 2XL,US,RG803 Black-Gray 2XL,280,0,0,SENWAYZON:US


In [22]:
# local_inv

In [23]:
local_inv.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,店铺
2881,水鞋-广州仓,SP001-406 Black 40,CA,SP001-406 Black 40,0,0,0,SEEKWAY:CA
2961,水鞋-广州仓,SP001-406 Black 40,US,SP001-406 Black 40,830,0,1618,SEEKWAY:US
3232,水鞋-广州仓,SP001-406 Black 40,共享,SP001-406 Black 40,0,0,0,共享
6271,水鞋-广州仓,SP001EU-406-Black-40,DE,SP001-406 Black 40,0,0,0,SENWAYZON:DE
9403,水鞋-广州仓,SP001-406 Black 40,JP,SP001-406 Black 40,0,0,0,SEEKWAY:JP
9915,水鞋-广州仓,SP001EU-406-Black-40-new,UK,SP001-406 Black 40,0,0,0,SENWAYZON:UK
14944,水鞋-广州仓,SP001EU-406-Black-41,DE,SP001-406 Black 40,0,0,40,SENWAYZON:DE
15304,水鞋-广州仓,SP001-406 Black New 40,CA,SP001-406 Black 40,95,92,0,SEEKWAY:CA
83800,TK本地仓,SP001-406 Black 40,TK本地仓,SP001-406 Black 40,0,0,104,TK本地仓:TK本地仓
88306,水鞋-广州仓,JPSP001-406 Black 40,JP,SP001-406 Black 40,25,0,0,SEEKWAY:JP


In [24]:
# local_inv.query('SKU == "SP001-406 Black 40"')

In [25]:
# 只包含海外仓（仓库 值只包含海外仓 的行）
sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓')]
sea_inv.仓库.unique()

array(['元坤海外仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'US易速达海外仓', 'UK商易海外仓', 'CN易速达:易速达美东GA仓',
       'IT永翔海外仓'], dtype=object)

In [26]:
# # 只包含海外仓（仓库 值只包含海外仓 的行）
# sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓|TK本地仓')]
# sea_inv.仓库.unique()

In [27]:
sea_inv

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,店铺
4816,元坤海外仓,WP005-BA102 Gray 40,共享,WP005-BA102 Gray 40,0,0,0,共享
4817,元坤海外仓,SA001-504 Black 43,共享,SA001-504 Black 43,0,0,0,共享
4818,元坤海外仓,WP005-BA071 Black 43,共享,WP005-BA071 Black 43,0,0,0,共享
4819,元坤海外仓,SP001-AB002 Dark Gray 44,US,SP001-AB002 Dark Gray 44,0,0,0,SEEKWAY:US
4820,元坤海外仓,SA001-505 Dark blue 44,共享,SA001-505 Dark blue 44,0,0,0,共享
...,...,...,...,...,...,...,...,...
105554,CN易速达:易速达美东GA仓,RG802 White S,共享,RG802 White S,0,0,0,共享
105555,CN易速达:易速达美东GA仓,RG802 White L,共享,RG802 White L,0,0,0,共享
105556,CN易速达:易速达美东GA仓,RG802 Gray M,共享,RG802 Gray M,0,0,0,共享
105829,US商易海外仓,RG802-CA Gray S,CA,RG802 Gray S,0,0,9,RIVMOUNT:CA


# 3、预处理预占单据

In [28]:
condition = (
    (yuzhan_df['单据状态'] == '审核通过') &
    (yuzhan_df['拣货状态'] == '已拣货') &
    (yuzhan_df['出库状态'] == '未出库')
)

yuzhan_df = yuzhan_df.loc[condition]

# 区分FBA和海外仓预占
condition = yuzhan_df['调入仓'].str.contains('FBA')

# FBA仓预占
yuzhan_fba = yuzhan_df[condition]

# 海外仓预占
yuzhan_sea = yuzhan_df[~condition]


yuzhan_fba['仓库'] = yuzhan_fba['调入仓'].str.split(pat='_', expand=True)[0]
yuzhan_fba['仓库'] = yuzhan_fba['仓库'].apply(lambda x: x.replace('EU', 'DE') if 'EU' in x else x)

yuzhan_fba = yuzhan_fba.pivot_table(index=['仓库','MSKU'], values=['调出数量'], aggfunc=sum).reset_index().rename(columns={'调出数量': 'FBA预占数'})

In [29]:
yuzhan_df

,SKU,MSKU,调出数量,调入仓,单据状态,拣货状态,出库状态
0,SWS002-A1213 Floral Glass 38-39,SWS002-1213 Floral Glass 38-39,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
1,SWS001-A1172 Watermarks 38-39,SWS001-1172 Watermarks 38-39,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
2,SWS005-503 circular blue 42-43,SWR005-503 circular blue 42-43,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
3,SWS002-A1501 Begonia flower 42-43,SWS002-A1501 Begonia flower 42-43,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
4,SWS001-A1421 Sead Paisley 38-39,SWS001-A1421 Sead Paisley 38-39,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
5,SWS001-A1542 Rain Forest 36-37,SWS001-A1542 Rain Forest 36-37,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库
6,SWS001-A1021 Love Rainbow 40-41,SWS001-1021 Love Rainbow 40-41,0,RIVMOUNT:US_FBA,审核通过,已拣货,未出库


In [30]:
yuzhan_fba

,仓库,MSKU,FBA预占数
0,RIVMOUNT:US,SWR005-503 circular blue 42-43,0
1,RIVMOUNT:US,SWS001-1021 Love Rainbow 40-41,0
2,RIVMOUNT:US,SWS001-1172 Watermarks 38-39,0
3,RIVMOUNT:US,SWS001-A1421 Sead Paisley 38-39,0
4,RIVMOUNT:US,SWS001-A1542 Rain Forest 36-37,0
5,RIVMOUNT:US,SWS002-1213 Floral Glass 38-39,0
6,RIVMOUNT:US,SWS002-A1501 Begonia flower 42-43,0


In [31]:
yuzhan_fba.columns

Index(['仓库', 'MSKU', 'FBA预占数'], dtype='object')

# 4、合并数据

## 4.1 匹配本地锁仓库存和FBA预占数

In [32]:
 # 筛选出一个不包含 “共享” 店铺的数据框，也就是筛选出‘本地锁仓数’
local_inv_lock = local_inv[~local_inv['店铺'].str.contains('共享')]

# 按照 “店铺”、“SKU” 和 “MSKU” 分组，并计算出每个分组 “本地” 列总和的新数据框
local_inv_lock_piv = local_inv_lock.pivot_table(index=['店铺', 'SKU','MSKU'], values='本地', aggfunc='sum').reset_index()

# FBA库存匹配本地锁仓库存（FBA库存为主表）
new_inv = pd.merge(left=fba_process, right=local_inv_lock_piv, left_on=['仓库', 'SKU','MSKU'], right_on=['店铺', 'SKU','MSKU'], how='left')
new_inv.drop(columns=['店铺'], inplace=True)
new_inv.rename(columns={'本地': '本地锁仓数'}, inplace=True)     #'可用量' -> '本地' -> '本地锁仓数'

# 匹配FBA预占数
new_inv = pd.merge(left=new_inv, right=yuzhan_fba, left_on=['仓库', 'MSKU'], right_on=['仓库', 'MSKU'], how='left')

In [33]:
new_inv.head(10)

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数
0,rivbos:US,003-1 sky blue,RBK003-1 sky blue,0,0,2,1,11,14,0.86,0.67,0.37,US,0,24.0,NaN
1,rivbos:US,003-1 white,RBK003-1 white,0,0,0,0,5,5,0.29,0.13,0.07,US,0,51.0,NaN
2,rivbos:US,003-2 Black Ice Green Lens,RBK003-2 Black Ice Green Lens,0,0,10,14,276,300,13.86,10.40,6.73,US,0,347.0,NaN
3,rivbos:US,003-2 Blue,RBK003-2 Blue,0,0,0,0,0,0,0.00,0.00,0.00,US,0,51.0,NaN
4,rivbos:US,004-1 Black,RBK004-1 Black,0,0,0,0,0,0,0.00,0.00,0.00,US,0,0.0,NaN
5,rivbos:US,004-1 Blue,RBK004-1 Blue,0,0,0,0,0,0,NaN,NaN,NaN,US,0,NaN,NaN
6,rivbos:US,004-2 Black Ice Red Lens,RBK004-2 Black Ice Red Lens,0,0,23,21,470,514,15.57,10.60,7.90,US,0,418.0,NaN
7,rivbos:US,004-2 Blue&blue,RBK004-2 W Blueblue,300,0,30,50,1448,1528,43.00,35.07,27.97,US,0,400.0,NaN
8,rivbos:US,004-826-white,RBK004-2 W White,0,0,0,0,0,0,0.00,0.00,0.00,US,0,0.0,NaN
9,rivbos:US,004-Wayfarer Black,RBK004-2 W Black,3180,74,711,78,3208,3997,110.57,95.67,79.00,US,0,3052.0,NaN


In [34]:
# local_inv_lock.to_excel('店铺不包含共享数.xlsx',index = False)

In [35]:
# local_inv_lock_piv.to_excel('不包含共享数.xlsx',index = False)

In [36]:
fba_process.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存'],
      dtype='object')

In [37]:
local_inv_lock_piv.query('MSKU == "SP001-406 Black 40"')

,店铺,SKU,MSKU,本地
4817,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0
5410,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0
7206,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,1618
16879,TK本地仓:TK本地仓,SP001-406 Black 40,SP001-406 Black 40,104


In [38]:
local_inv_lock_piv

,店铺,SKU,MSKU,本地
0,CIKERWEL:US,832-1 BlackGrey,832-1 Black&Grey,0
1,CIKERWEL:US,KSK10011-Snowflake Blue 22-23,KSK10011-Snowflake Blue 22-23-US,10
2,CIKERWEL:US,KSK10011-Snowflake Blue 24-25,KSK10011-Snowflake Blue 24-25-US,10
3,CIKERWEL:US,KSK10011-Snowflake Blue 26-27,KSK10011-Snowflake Blue 26-27-US,20
4,CIKERWEL:US,KSK10011-Snowflake Blue 28-29,KSK10011-Snowflake Blue 28-29-US,30
...,...,...,...,...
21616,seekwayer:United_States,SMRG902 Pink L,SG902-Pink-L,0
21617,seekwayer:United_States,SMRG902 Pink M,SG902-Pink-M,0
21618,seekwayer:United_States,SMRG902 Pink S,SG902-Pink-S,0
21619,seekwayer:United_States,SMRG902 Pink XL,SG902-Pink-XL,0


In [39]:
new_inv.query('SKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数
15631,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,77,77,0.86,0.8,0.6,US,0,3.0,NaN
18593,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,NaN,NaN,NaN,CA,0,NaN,NaN


In [40]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存', '本地锁仓数',
       'FBA预占数'],
      dtype='object')

In [41]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存', '本地锁仓数',
       'FBA预占数'],
      dtype='object')

In [42]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数
15631,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,77,77,0.86,0.8,0.6,US,0,3.0,NaN
18593,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,NaN,NaN,NaN,CA,0,NaN,NaN


In [43]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数
15631,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,77,77,0.86,0.8,0.6,US,0,3.0,NaN
18593,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,NaN,NaN,NaN,CA,0,NaN,NaN


In [44]:
local_inv_lock.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,店铺
15022,水鞋-广州仓,SP001-415 Dark Grey 47,US,SP001-415 Dark Grey 47,30,0,3,SEEKWAY:US
83911,TK本地仓,SP001-415 Dark Grey 47,TK本地仓,SP001-415 Dark Grey 47,0,0,0,TK本地仓:TK本地仓


## 4.2 匹配本地共享数据


In [45]:
# 采购数据，按照 “SKU”分组，并计算出每个分组 '已下单数量', '本地-在途' 列总和的新数据框。'本地-在途'（待质检数）
# todo
procurement_documents = local_inv.groupby(by=['店铺','SKU','MSKU'])[['已下单数量', '本地-在途']].sum().reset_index()
# procurement_documents = local_inv.groupby(by=['SKU'])[['已下单数量', '本地-在途']].sum().reset_index()

# 本地-共享
share_local = local_inv[local_inv['店铺'] == '共享'].groupby(by=['SKU'])['本地'].sum().reset_index().rename(columns={'本地': '本地-共享'})

In [46]:
procurement_documents.query('SKU == "SP001-415 Dark Grey 47"')

,店铺,SKU,MSKU,已下单数量,本地-在途
7307,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,30,0
16990,TK本地仓:TK本地仓,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0
25201,共享,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0


In [47]:
share_local.query('SKU == "SP001-415 Dark Grey 47"')

,SKU,本地-共享
3580,SP001-415 Dark Grey 47,0


In [48]:
procurement_documents.rename(columns={'店铺':'仓库'}, inplace=True)

procurement_documents.columns

Index(['仓库', 'SKU', 'MSKU', '已下单数量', '本地-在途'], dtype='object')

In [49]:
procurement_documents.仓库.unique()

array(['CIKERWEL:US', 'GLOW:CA', 'GLOW:US', 'Mr_y:US', 'RIVMOUNT:CA',
       'RIVMOUNT:US', 'ROUNDYAP:US', 'SAYOLA:CA', 'SAYOLA:US',
       'SEEKWAY:CA', 'SEEKWAY:JP', 'SEEKWAY:US', 'SENWAYZON:CA',
       'SENWAYZON:DE', 'SENWAYZON:UK', 'SENWAYZON:US',
       'SIMARI-:United_States', 'SIMARI:CA', 'SIMARI:DE', 'SIMARI:JP',
       'SIMARI:UK', 'TK本地仓:TK本地仓', 'WAYFINDING:CA', 'WAYFINDING:US',
       'rivbos:CA', 'rivbos:DE', 'rivbos:JP', 'rivbos:UK', 'rivbos:US',
       'seekwayer:United_States', '共享'], dtype=object)

In [50]:
dfs = [new_inv, share_local]

# 使用 reduce 函数进行左连接
merged_df = reduce(lambda left, right: pd.merge(left, right, on='SKU', how='left'), dfs)

In [51]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存', '本地锁仓数',
       'FBA预占数'],
      dtype='object')

In [52]:
new_inv.仓库.unique()   # 此数据框的仓库实际是店铺：站点

array(['rivbos:US', 'rivbos:CA', 'rivbos:UK', 'rivbos:JP', 'rivbos:DE',
       'SEEKWAY:US', 'SEEKWAY:CA', 'SEEKWAY:JP', 'SIMARI:JP',
       'SENWAYZON:US', 'SENWAYZON:CA', 'SENWAYZON:UK', 'SENWAYZON:DE',
       'SAYOLA:US', 'SAYOLA:CA', 'RIVMOUNT:US', 'RIVMOUNT:CA',
       'SIMARI:US', 'SIMARI:CA', 'SIMARI:UK', 'SIMARI:DE', 'ROUNDYAP:US',
       'Mr_y:US', 'GLOW:US', 'CIKERWEL:US', 'WAYFINDING:US',
       'WAYFINDING:CA', 'GLOW:CA', 'CIKERWEL:CA'], dtype=object)

In [53]:
local_inv.店铺.unique() 

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW:US',
       'RIVMOUNT:US', 'ROUNDYAP:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'SENWAYZON:US', 'SAYOLA:US', 'SENWAYZON:CA',
       'WAYFINDING:US', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING:CA',
       'SIMARI:CA', 'GLOW:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [54]:
local_inv.仓库.unique() 

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '棉帽-广州仓', '其它品类-广州仓', 'TK本地仓'],
      dtype=object)

In [55]:
procurement_documents.columns

Index(['仓库', 'SKU', 'MSKU', '已下单数量', '本地-在途'], dtype='object')

In [56]:
share_local.columns

Index(['SKU', '本地-共享'], dtype='object')

In [57]:
merged_df.query('SKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数,本地-共享
15631,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,77,77,0.86,0.8,0.6,US,0,3.0,NaN,0.0
18593,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,NaN,NaN,NaN,CA,0,NaN,NaN,0.0


In [58]:
merged_df = pd.merge(merged_df, procurement_documents, on = ['仓库', 'SKU', 'MSKU'], how = 'left')

In [59]:
merged_df.query('SKU == "SK001-701 Stripe Black 44-45"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数,本地-共享,已下单数量,本地-在途
1534,SEEKWAY:US,SK001-701 Stripe Black 44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0.00,0.00,0.00,US,0,0.0,NaN,0.0,0.0,0.0
1962,SEEKWAY:US,SK002-Stripe Black 701 44-45,SK001-701 Stripe Black 44-45,0,0,47,475,4323,4845,40.57,62.33,59.57,US,0,8206.0,NaN,0.0,1400.0,0.0
2603,SEEKWAY:CA,SK001-701 Stripe Black 44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0.00,0.00,0.00,CA,0,0.0,NaN,0.0,0.0,0.0
3082,SEEKWAY:CA,SK002-Stripe Black 701 44-45,SK001-701 Stripe Black 44-45,172,0,1,8,144,153,2.14,2.73,2.83,CA,0,0.0,NaN,0.0,250.0,0.0
5689,SENWAYZON:UK,SK001EU-701-Stripe-Black-44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0.00,0.00,0.00,UK,0,0.0,NaN,0.0,0.0,0.0
6270,SENWAYZON:DE,SK001EU-701-Stripe-Black-44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0.00,0.00,0.00,DE,0,0.0,NaN,0.0,0.0,0.0


# 4.3 数据透视本地库存

In [60]:
# product_local = product_df.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()
product_local = local_inv.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()


piv_df1 = pd.concat(objs=[product_local['SKU'], product_local['本地']], axis=1)
piv_df2 = local_inv.pivot_table(index='SKU', values=['本地-在途'], aggfunc=sum, fill_value=0).reset_index()

In [61]:
local_piv = pd.merge(left=piv_df1, right=piv_df2, on='SKU', how='left')

In [62]:
local_piv.query('SKU == "SP001-406 Black 40"')

,SKU,CA,DE,JP,TK本地仓,UK,US,United_States,共享,本地-在途
3754,SP001-406 Black 40,0,40,0,104,0,1618,0,0,92


In [63]:
local_piv

,SKU,CA,DE,JP,TK本地仓,UK,US,United_States,共享,本地-在途
0,301 Black,0,0,1200,0,0,0,0,0,0
1,301 Blue,0,0,250,0,0,0,0,0,0
2,301 Camel,0,0,0,0,0,0,0,0,0
3,301 Dimgray,0,0,550,0,0,0,0,0,0
4,301 Gray,0,0,300,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
7284,YB001 Grey 65cm,0,0,0,0,0,0,0,0,0
7285,YB001 Grey 75cm,0,0,0,0,0,0,0,0,0
7286,YB001 Pink 55cm,0,0,0,0,0,0,0,0,0
7287,YB001 Pink 65cm,0,0,0,0,0,0,0,0,0


# 5、处理海外仓库存

In [64]:
# 构建海外仓站点映射
site_map = {'元坤海外仓': 'US', '九方欧洲海外仓': 'DE', 'CA仓搜海外仓' : 'CA', 
            'DE延讯海外仓':'DE', 'UK延讯海外仓' : 'UK', 'JP永翔海外仓' : 'JP',
            'DE商易海外仓' : 'DE', 'US商易海外仓' : 'US', 'US易速达海外仓': 'US', 
            'UK商易海外仓' : 'UK', 'CN易速达:易速达美东GA仓':'US', 'IT永翔海外仓':'IT'}

In [65]:
sea_inv['仓库'].unique()

array(['元坤海外仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'US易速达海外仓', 'UK商易海外仓', 'CN易速达:易速达美东GA仓',
       'IT永翔海外仓'], dtype=object)

In [66]:
if len(site_map) != sea_inv['仓库'].unique().size:
    raise ValueError(f"海外仓数量有变动，请重新维护变量：site_map ")

In [67]:
# 为海外仓库 映射站点
sea_inv['站点'] = sea_inv['仓库'].apply(site_map.get)

In [68]:
# 海外仓库存全部归为共享
temp_df = sea_inv[['站点','SKU', '仓库', '本地']].pivot_table(index=['站点','SKU'], columns=['仓库'], aggfunc=sum).fillna(0).reset_index()
sea_inv = pd.concat(objs=[temp_df['站点'], temp_df['SKU'], temp_df['本地']], axis=1)


In [69]:
# 预处理海外仓-预占数据
yuzhan_sea['站点'] = yuzhan_sea['调入仓'].apply(site_map.get)

yuzhan_sea = yuzhan_sea.pivot_table(index=['站点', 'SKU'], columns=['调入仓'], values='调出数量', aggfunc=sum).reset_index()

# 重命名字段
columns = yuzhan_sea.columns
new_columns = columns[:2].to_list() + [f'{name}_预占' for name in columns[2:]]
yuzhan_sea.columns = new_columns
if yuzhan_sea.size == 0:
    print('本周无海外仓预占数据')

sea_ret = pd.merge(left=sea_inv, right=yuzhan_sea, on=['站点', 'SKU'], how='outer')

# 填充空值
numeric_columns = sea_ret.select_dtypes(include='number').columns
sea_ret[numeric_columns] = sea_ret[numeric_columns].fillna(0)

本周无海外仓预占数据


# 最后的数据处理与输出

In [70]:
merged_df.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,SKU,已发货(数量),正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,站点,长期仓储库存,本地锁仓数,FBA预占数,本地-共享,已下单数量,本地-在途
2062,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,0,0,35,10,1171,1216,17.00,16.27,14.57,US,0,1618.0,NaN,0.0,830.0,0.0
3187,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,CA,0,0.0,NaN,0.0,0.0,0.0
3914,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,JP,0,0.0,NaN,0.0,0.0,0.0
3915,SEEKWAY:JP,SP001-406 Black 40 NEW,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,JP,0,NaN,NaN,0.0,NaN,NaN
6569,SENWAYZON:DE,SP001EU-406-Black-41,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,DE,0,40.0,NaN,0.0,0.0,0.0
14346,SENWAYZON:UK,SP001EU-406-Black-40-new,SP001-406 Black 40,0,0,0,0,0,0,0.00,0.00,0.00,UK,0,0.0,NaN,0.0,0.0,0.0
15123,SEEKWAY:CA,SP001-406 Black New 40,SP001-406 Black 40,59,0,21,42,72,135,2.00,1.33,1.10,CA,0,0.0,NaN,0.0,95.0,92.0
19102,SEEKWAY:JP,JPSP001-406 Black 40,SP001-406 Black 40,0,0,0,0,40,40,0.43,0.20,0.20,JP,0,0.0,NaN,0.0,25.0,0.0


In [71]:
# 填充缺失值
numeric_columns = merged_df.select_dtypes(include='number').columns

merged_df[numeric_columns] = merged_df[numeric_columns].fillna(0)

In [72]:
merged_df['FBA在途'] = merged_df['FBA预占数'] + merged_df['已发货(数量)'] + merged_df['正在接收(数量)']

In [73]:
merged_df.columns

Index(['仓库', 'MSKU', 'SKU', '已发货(数量)', '正在接收(数量)', '预留-运营中心转运', '预留-运营中心正在处理',
       'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '站点', '长期仓储库存', '本地锁仓数',
       'FBA预占数', '本地-共享', '已下单数量', '本地-在途', 'FBA在途'],
      dtype='object')

In [74]:
# 调整字段输出顺序
new_order = ['站点', '仓库', 'SKU', 'MSKU', 
                'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
                '预留-运营中心转运','预留-运营中心正在处理', 'FBA可售', '可用库存', 
                '长期仓储库存', 
                '本地锁仓数', '本地-共享', '已下单数量', '本地-在途',
                '7天日均', '15天日均', '30天日均']
merged_df = merged_df[new_order]

merged_df.rename(columns={'可用库存': 'FBA可用库存'}, inplace=True)

In [75]:
merged_df.columns

Index(['站点', '仓库', 'SKU', 'MSKU', 'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
       '预留-运营中心转运', '预留-运营中心正在处理', 'FBA可售', 'FBA可用库存', '长期仓储库存', '本地锁仓数',
       '本地-共享', '已下单数量', '本地-在途', '7天日均', '15天日均', '30天日均'],
      dtype='object')

In [76]:
# 指定要添加颜色的列
columns_to_color = {'站点': 'FFF07C82', '仓库': 'FFF07C82', 'SKU': 'FFF07C82', 'MSKU': 'FFF07C82', 
                'FBA预占数': 'FF40A070', '已发货(数量)': 'FF40A070', '正在接收(数量)': 'FF40A070', 'FBA在途': 'FF40A070',
                '预留-运营中心转运': 'FF9ABEAF','预留-运营中心正在处理': 'FF9ABEAF', 'FBA可售': 'FF9ABEAF', 'FBA可用库存': 'FF9ABEAF', 
                '长期仓储库存': 'FF9ABEAF', 
                '本地锁仓数': 'FFE2D849', '本地-共享': 'FFE2D849', '已下单数量': 'FFE2D849', '本地-在途': 'FFE2D849',
                '7天日均': 'FFE16723', '15天日均': 'FFE16723', '30天日均': 'FFE16723'}

# 将 DataFrame 转换为 Excel 格式，并创建一个 Pandas Excel writer
with pd.ExcelWriter(f'../src_data/处理后的库存/库存{date_parm}.xlsx', engine='openpyxl') as writer:
    merged_df.to_excel(writer, sheet_name='库存汇总', index=False)
    local_piv.to_excel(writer, sheet_name='本地仓库存透视', index=False)
    sea_ret.to_excel(writer, sheet_name='海外仓库存', index=False)
    

    # 获取 Pandas Excel writer 中的 workbook 和 worksheet
    workbook = writer.book
    worksheet = writer.sheets['库存汇总']

    # 使用 Styler 对象为指定列的 header 行添加样式
    rows = merged_df.shape[0]
    for column, color in columns_to_color.items():
        col_idx = merged_df.columns.get_loc(column) + 1
        # print(col_idx)
        for index in range(1, rows+2):
            header_cell = worksheet.cell(index, col_idx)
            header_cell.fill = openpyxl.styles.PatternFill(start_color=color, end_color=color, fill_type='solid')

# 打印结果
print("Excel 文件已生成")


Excel 文件已生成
